# Accessible RetNet — 124M Parameter Language Model
> **Paper:** *Retentive Network: A Successor to Transformer for Large Language Models* (Sun et al., 2023)  
> **Target:** Train on WikiText-103, achieve validation perplexity ~15.2  
> **Architecture:** d_model=768, 12 layers, 6 heads, xPos encoding, multi-scale retention  

---

### What this notebook covers — in order:
1. **Environment** — Install PyTorch + CUDA, verify GPU  
2. **Hyperparameters** — All config in one place  
3. **Model Architecture** — xPos, Multi-Scale Retention, RetNet blocks, full LM  
4. **Dataset** — WikiText-103 download, GPT-2 tokenization, 512-token chunking  
5. **Training Setup** — AdamW, warmup-cosine LR, FP16, gradient accumulation  
6. **Training Loop** — Full 50k-step run with live metrics  
7. **Evaluation** — Val/test perplexity  
8. **Generation** — Recurrent O(1) inference, temperature/top-p sampling  

> Run cells **top to bottom** in sequence. Each section builds on the previous.

---
## 1. Environment Setup
Installs PyTorch 2.1.2 with bundled CUDA 11.8 (RTX 2070 / Colab T4 compatible).  
**On Google Colab**: CUDA is pre-installed — the pip install still works fine.  
**On local GPU**: RTX 2070 is Compute Capability 7.5, fully supported.

In [ ]:
# Install all dependencies
# On Colab: this cell ensures the correct PyTorch+CUDA version
# On local: run 'pip install -r requirements.txt' instead if preferred

import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

# PyTorch 2.1.2 with bundled CUDA 11.8
pip(
    "torch==2.1.2+cu118",
    "torchvision==0.16.2+cu118",
    "torchaudio==2.1.2+cu118",
    "--index-url", "https://download.pytorch.org/whl/cu118"
)

# Other dependencies
pip("transformers>=4.38.0", "datasets>=2.18.0", "tokenizers>=0.15.0")
pip("numpy<2", "tqdm>=4.66.0")  # numpy<2 required for PyTorch 2.1.2 compatibility

print("All packages installed.")

In [ ]:
# Verify GPU / CUDA setup
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    total_gb = props.total_memory / 1024**3
    free_gb  = (props.total_memory - torch.cuda.memory_allocated(0)) / 1024**3
    cc = f"{props.major}.{props.minor}"
    print(f"GPU             : {name}")
    print(f"VRAM            : {total_gb:.1f} GB total | {free_gb:.1f} GB free")
    print(f"Compute Cap.    : {cc}")

    # Quick FP16 smoke test
    a = torch.randn(128, 128, dtype=torch.float16, device='cuda')
    b = torch.matmul(a, a)
    assert not torch.isnan(b).any(), "FP16 matmul produced NaN!"
    print("FP16 matmul     : OK")
else:
    print("WARNING: No GPU detected. Training will be very slow on CPU.")

---
## 2. Hyperparameters & Configuration
All tunable settings in one place. Change values here and re-run to experiment.

In [ ]:
# ============================================================
#  MODEL ARCHITECTURE HYPERPARAMETERS
# ============================================================
D_MODEL      = 768      # Hidden dimension
N_LAYERS     = 12       # Number of RetNet blocks
N_HEADS      = 6        # Number of retention heads
D_HEAD_QK    = 128      # Q and K head dimension (asymmetric)
D_HEAD_V     = 256      # V head dimension (asymmetric)
D_FFN        = 1536     # Feed-forward expansion dimension
VOCAB_SIZE   = 50257    # GPT-2 vocabulary
MAX_SEQ_LEN  = 512      # Maximum sequence length
DROPOUT      = 0.0      # Dropout (0 = disabled for this scale)

# Multi-scale decay rates: gamma_h = 1 - 2^(-(5+h)) for h in {0,...,5}
# Head 0 (fastest decay): gamma = 0.96875
# Head 5 (slowest decay): gamma = 0.99902
GAMMAS = [1.0 - 2.0 ** (-(5 + h)) for h in range(N_HEADS)]

# ============================================================
#  TRAINING HYPERPARAMETERS
# ============================================================
TOTAL_STEPS    = 50_000   # Total gradient update steps
WARMUP_STEPS   = 1_000    # Linear LR warmup steps
MAX_LR         = 3e-4     # Peak learning rate
MIN_LR_RATIO   = 0.1      # min_lr = MAX_LR * MIN_LR_RATIO (cosine floor)
ADAM_BETA1     = 0.9
ADAM_BETA2     = 0.98
ADAM_EPS       = 1e-8
WEIGHT_DECAY   = 0.05
GRAD_CLIP      = 1.0      # Gradient norm clipping

# ============================================================
#  BATCH / MEMORY HYPERPARAMETERS
# ============================================================
PHYS_BATCH     = 8        # Physical batch size per GPU step (8GB VRAM)
ACCUM_STEPS    = 4        # Gradient accumulation -> effective batch = 32
EFFECTIVE_BATCH = PHYS_BATCH * ACCUM_STEPS

# ============================================================
#  LOGGING / CHECKPOINTING
# ============================================================
LOG_EVERY      = 50       # Print metrics every N steps
EVAL_EVERY     = 2_000    # Run validation every N steps
SAVE_EVERY     = 2_000    # Save checkpoint every N steps
CHECKPOINT_DIR = "./checkpoints"

# ============================================================
#  GENERATION DEFAULTS
# ============================================================
GEN_TEMPERATURE    = 0.8
GEN_TOP_P          = 0.9
GEN_REP_PENALTY    = 1.3
GEN_NGRAM_BLOCK    = 3
GEN_MAX_NEW_TOKENS = 200

print("=" * 50)
print("  MODEL CONFIG")
print("=" * 50)
print(f"  d_model    : {D_MODEL}")
print(f"  n_layers   : {N_LAYERS}")
print(f"  n_heads    : {N_HEADS}  (gammas: {[round(g,5) for g in GAMMAS]})")
print(f"  d_qk / d_v : {D_HEAD_QK} / {D_HEAD_V}  (asymmetric)")
print(f"  d_ffn      : {D_FFN}")
print(f"  vocab_size : {VOCAB_SIZE}")
print(f"  seq_len    : {MAX_SEQ_LEN}")
print()
print("  TRAINING CONFIG")
print("=" * 50)
print(f"  steps      : {TOTAL_STEPS:,}  (warmup: {WARMUP_STEPS:,})")
print(f"  LR         : {MAX_LR} -> cosine -> {MAX_LR*MIN_LR_RATIO}")
print(f"  optimizer  : AdamW (b1={ADAM_BETA1}, b2={ADAM_BETA2}, wd={WEIGHT_DECAY})")
print(f"  batch      : {PHYS_BATCH} x {ACCUM_STEPS} accum = {EFFECTIVE_BATCH} effective")
print(f"  grad clip  : {GRAD_CLIP}")

---
## 3. Model Architecture

### Key architectural decisions:
- **xPos encoding** — Rotary-style with extrapolation decay. Computed in float32 even during FP16 training to prevent underflow. Decay exponent bounded to `[0, 1]` by dividing position by `scale_base=512`.
- **Multi-Scale Retention** — Each head has its own decay γ_h. Parallel mode for training (full causal matrix), recurrent mode for inference (O(1) state per step).
- **Asymmetric heads** — Q,K use 128-dim, V uses 256-dim. This gives richer value representations.
- **GroupNorm + SiLU gate** — On concatenated head outputs before projection.
- **Weight tying** — Token embedding matrix = LM head matrix (saves ~39M params).

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from typing import Optional, List, Tuple

# ────────────────────────────────────────────────────────────
#  RetNet Configuration
# ────────────────────────────────────────────────────────────

class RetNetConfig:
    """All architecture hyperparameters in one object."""
    def __init__(
        self,
        d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
        d_head_qk=D_HEAD_QK, d_head_v=D_HEAD_V, d_ffn=D_FFN,
        vocab_size=VOCAB_SIZE, max_seq_len=MAX_SEQ_LEN, dropout=DROPOUT
    ):
        self.d_model     = d_model
        self.n_layers    = n_layers
        self.n_heads     = n_heads
        self.d_head_qk   = d_head_qk   # Q, K head dimension
        self.d_head_v    = d_head_v    # V head dimension (asymmetric)
        self.d_ffn       = d_ffn
        self.vocab_size  = vocab_size
        self.max_seq_len = max_seq_len
        self.dropout     = dropout
        # Multi-scale decay: gamma_h = 1 - 2^(-(5+h)) for h in {0,...,n_heads-1}
        self.gammas = [1.0 - 2.0 ** (-(5 + h)) for h in range(n_heads)]

print("RetNetConfig defined.")

In [ ]:
# ────────────────────────────────────────────────────────────
#  xPos: Extrapolatable Positional Encoding
# ────────────────────────────────────────────────────────────
#
#  xPos combines rotary embeddings with a per-dimension decay scale ζ.
#  The decay keeps long-range positions from dominating short-range ones,
#  enabling extrapolation beyond the training sequence length.
#
#  FP16 safety note: zeta^pos can underflow to 0 in FP16 for large pos.
#  Fix: use zeta^(pos/scale_base) so the exponent stays in [0, 1].
#  All computations are cast to float32 and converted back at the end.

def _build_xpos_decay(d: int) -> torch.Tensor:
    """
    Per-dimension decay scale: zeta_i = (i + d/4) / d
    Values in [0.5, 1.0). Shape: (d//2,)
    """
    half = d // 2
    idx = torch.arange(half, dtype=torch.float32)
    return (idx + half) / d   # in [0.5, 1.0)


def apply_xpos(x: torch.Tensor, offset: int = 0, scale_base: int = 512) -> torch.Tensor:
    """
    Apply xPos to queries. Shape: (B, T, H, d_head) -> same.
    Uses zeta^(pos/scale_base) for FP16 safety.
    """
    orig_dtype = x.dtype
    B, T, H, d = x.shape
    half = d // 2

    # Always compute in float32 to avoid FP16 underflow
    pos = torch.arange(offset, offset + T, dtype=torch.float32, device=x.device)  # (T,)
    inv_freq = 1.0 / (10000.0 ** (torch.arange(half, dtype=torch.float32, device=x.device) / half))
    angles = torch.outer(pos, inv_freq)  # (T, half)

    # xPos decay: zeta^(pos/scale_base), bounded in [zeta, 1.0] ⊆ [0.5, 1.0]
    zeta = _build_xpos_decay(d).to(x.device)       # (half,)
    decay = torch.pow(zeta.unsqueeze(0), (pos / scale_base).unsqueeze(1))  # (T, half)

    cos_a = (torch.cos(angles) * decay).unsqueeze(0).unsqueeze(2)  # (1, T, 1, half)
    sin_a = (torch.sin(angles) * decay).unsqueeze(0).unsqueeze(2)

    x_f32 = x.float()
    x1, x2 = x_f32[..., :half], x_f32[..., half:]
    return torch.cat([x1 * cos_a - x2 * sin_a,
                      x1 * sin_a + x2 * cos_a], dim=-1).to(orig_dtype)


def apply_xpos_key(x: torch.Tensor, offset: int = 0, scale_base: int = 512) -> torch.Tensor:
    """
    Apply xPos to keys (inverse decay: zeta^(-pos/scale_base)).
    Keys use the conjugate rotation so Q·K inner product is position-aware.
    """
    orig_dtype = x.dtype
    B, T, H, d = x.shape
    half = d // 2

    pos = torch.arange(offset, offset + T, dtype=torch.float32, device=x.device)
    inv_freq = 1.0 / (10000.0 ** (torch.arange(half, dtype=torch.float32, device=x.device) / half))
    angles = torch.outer(pos, inv_freq)

    zeta = _build_xpos_decay(d).to(x.device)
    decay = torch.pow(zeta.unsqueeze(0), -(pos / scale_base).unsqueeze(1))  # inverse

    cos_a = (torch.cos(angles) * decay).unsqueeze(0).unsqueeze(2)
    sin_a = (torch.sin(angles) * decay).unsqueeze(0).unsqueeze(2)

    x_f32 = x.float()
    x1, x2 = x_f32[..., :half], x_f32[..., half:]
    return torch.cat([x1 * cos_a - x2 * sin_a,
                      x1 * sin_a + x2 * cos_a], dim=-1).to(orig_dtype)


print("xPos functions defined.")

In [ ]:
# ────────────────────────────────────────────────────────────
#  Multi-Scale Retention
# ────────────────────────────────────────────────────────────
#
#  PARALLEL MODE (training):
#    Retention(Q, K, V) = (Q @ K^T ⊙ D̃) @ V
#    D[i,j] = gamma^(i-j)  if i >= j  else 0
#    D̃[i,j] = D[i,j] / sqrt(sum_k D[i,k])  (row normalization)
#
#  RECURRENT MODE (inference, O(1) memory):
#    s_t = gamma * s_{t-1} + k_t^T ⊗ v_t
#    o_t = q_t @ s_t
#
#  The decay masks are pre-computed once at init and stored as
#  a buffer (H, T, T) so they auto-move to GPU with .to(device).

class MultiScaleRetention(nn.Module):

    def __init__(self, config: RetNetConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.d_model = config.d_model
        self.d_qk    = config.d_head_qk
        self.d_v     = config.d_head_v
        self.gammas  = config.gammas

        # Input projections (no bias for efficiency)
        self.q_proj    = nn.Linear(config.d_model, config.n_heads * config.d_head_qk, bias=False)
        self.k_proj    = nn.Linear(config.d_model, config.n_heads * config.d_head_qk, bias=False)
        self.v_proj    = nn.Linear(config.d_model, config.n_heads * config.d_head_v,  bias=False)
        self.gate_proj = nn.Linear(config.d_model, config.n_heads * config.d_head_v,  bias=False)
        self.out_proj  = nn.Linear(config.n_heads * config.d_head_v, config.d_model,  bias=False)

        # GroupNorm: one group per head, normalizes concatenated head outputs
        self.group_norm = nn.GroupNorm(config.n_heads, config.n_heads * config.d_head_v, eps=1e-5)

        # Pre-built causal decay masks — (H, T, T) float32 buffer
        D_all = self._build_all_decay_masks(config.max_seq_len, config.gammas)
        self.register_buffer("decay_mask_full", D_all)

        # Weight init
        for m in [self.q_proj, self.k_proj, self.v_proj, self.gate_proj, self.out_proj]:
            nn.init.xavier_uniform_(m.weight)

    # ── Decay mask pre-computation ──────────────────────────

    @staticmethod
    def _build_all_decay_masks(T: int, gammas: list) -> torch.Tensor:
        """
        Build all H causal decay matrices at once.
        D̃[h, i, j] = gamma_h^(i-j) / sqrt(row_sum)  for i >= j, else 0
        Returns (H, T, T) float32.
        """
        H = len(gammas)
        i_idx = torch.arange(T).float().unsqueeze(1)   # (T, 1)
        j_idx = torch.arange(T).float().unsqueeze(0)   # (1, T)
        diff  = (i_idx - j_idx).clamp(min=0)           # (T, T)
        causal = (i_idx >= j_idx).float()               # (T, T)

        gamma_t = torch.tensor(gammas, dtype=torch.float32).view(H, 1, 1)
        D = (gamma_t ** diff.unsqueeze(0)) * causal.unsqueeze(0)  # (H, T, T)

        row_sum = D.sum(dim=-1, keepdim=True).clamp(min=1e-6)
        return D / row_sum.sqrt()   # (H, T, T), row-normalized

    def _get_decay_mask(self, T: int, dtype) -> torch.Tensor:
        """Slice prebuilt mask to current sequence length T."""
        return self.decay_mask_full[:, :T, :T].to(dtype)  # (H, T, T)

    # ── Parallel mode (training) ────────────────────────────

    def forward_parallel(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, _ = x.shape
        dtype = x.dtype

        Q = self.q_proj(x).view(B, T, self.n_heads, self.d_qk)  # (B, T, H, d_qk)
        K = self.k_proj(x).view(B, T, self.n_heads, self.d_qk)
        V = self.v_proj(x).view(B, T, self.n_heads, self.d_v)

        Q = apply_xpos(Q)
        K = apply_xpos_key(K)

        # Rearrange to (B, H, T, d) for batched matmul
        Q = Q.permute(0, 2, 1, 3).contiguous()  # (B, H, T, d_qk)
        K = K.permute(0, 2, 1, 3).contiguous()
        V = V.permute(0, 2, 1, 3).contiguous()  # (B, H, T, d_v)

        # Scaled dot-product over all heads simultaneously
        scores = torch.matmul(Q, K.transpose(-2, -1)) * (self.d_qk ** -0.5)  # (B, H, T, T)

        # Apply causal decay mask (broadcast over batch)
        D = self._get_decay_mask(T, dtype)   # (H, T, T)
        scores = scores * D.unsqueeze(0)      # (B, H, T, T)

        out = torch.matmul(scores, V)                              # (B, H, T, d_v)
        out = out.permute(0, 2, 1, 3).contiguous()                 # (B, T, H, d_v)
        concat = out.view(B, T, self.n_heads * self.d_v)           # (B, T, H*d_v)

        # GroupNorm expects (B, C, T)
        concat_normed = self.group_norm(concat.permute(0, 2, 1)).permute(0, 2, 1)

        gate = F.silu(self.gate_proj(x))   # (B, T, H*d_v)
        return self.out_proj(concat_normed * gate)  # (B, T, d_model)

    # ── Recurrent mode (inference, O(1) memory) ─────────────

    def forward_recurrent(
        self,
        x: torch.Tensor,
        states: Optional[List[torch.Tensor]],
        position: int = 0,
    ) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        """
        Single-step recurrent inference.
        x:      (B, 1, d_model)  — one token
        states: list of H tensors, each (B, d_qk, d_v)  — per-head state s_t
        """
        B = x.shape[0]
        device, dtype = x.device, x.dtype

        Q = self.q_proj(x).view(B, 1, self.n_heads, self.d_qk)   # (B, 1, H, d_qk)
        K = self.k_proj(x).view(B, 1, self.n_heads, self.d_qk)
        V = self.v_proj(x).view(B, 1, self.n_heads, self.d_v)

        Q = apply_xpos(Q, offset=position)
        K = apply_xpos_key(K, offset=position)

        Q = Q.squeeze(1)   # (B, H, d_qk) — no time dim needed in recurrent mode
        K = K.squeeze(1)
        V = V.squeeze(1)   # (B, H, d_v)

        if states is None:
            states = [torch.zeros(B, self.d_qk, self.d_v, device=device, dtype=dtype)
                      for _ in range(self.n_heads)]

        new_states, head_outputs = [], []
        for h in range(self.n_heads):
            q_h, k_h, v_h = Q[:, h], K[:, h], V[:, h]   # each (B, d_qk or d_v)
            gamma = self.gammas[h]
            # State update: s_t = gamma * s_{t-1} + k_t^T ⊗ v_t
            s_t = gamma * states[h] + torch.bmm(k_h.unsqueeze(2), v_h.unsqueeze(1))  # (B, d_qk, d_v)
            new_states.append(s_t)
            # Output: o_t = q_t @ s_t
            o_t = torch.bmm(q_h.unsqueeze(1), s_t).squeeze(1)   # (B, d_v)
            head_outputs.append(o_t)

        concat = torch.cat(head_outputs, dim=-1).unsqueeze(1)   # (B, 1, H*d_v)
        concat_normed = self.group_norm(concat.permute(0, 2, 1)).permute(0, 2, 1)
        gate = F.silu(self.gate_proj(x.squeeze(1))).unsqueeze(1)
        out  = self.out_proj(concat_normed * gate)               # (B, 1, d_model)
        return out, new_states

    def forward(self, x, recurrent_states=None, position=0):
        if recurrent_states is not None or x.shape[1] == 1:
            return self.forward_recurrent(x, recurrent_states, position)
        return self.forward_parallel(x)


print("MultiScaleRetention defined.")

In [ ]:
# ────────────────────────────────────────────────────────────
#  Feed-Forward Network
# ────────────────────────────────────────────────────────────

class FeedForward(nn.Module):
    """Two-layer GELU FFN: d_model -> d_ffn -> d_model."""
    def __init__(self, config: RetNetConfig):
        super().__init__()
        self.fc1 = nn.Linear(config.d_model, config.d_ffn)
        self.fc2 = nn.Linear(config.d_ffn,  config.d_model)
        nn.init.xavier_uniform_(self.fc1.weight); nn.init.zeros_(self.fc1.bias)
        nn.init.xavier_uniform_(self.fc2.weight); nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


# ────────────────────────────────────────────────────────────
#  RetNet Block
# ────────────────────────────────────────────────────────────
#
#  x -> LayerNorm -> MultiScaleRetention -> + residual
#    -> LayerNorm -> FeedForward          -> + residual

class RetNetBlock(nn.Module):
    def __init__(self, config: RetNetConfig):
        super().__init__()
        self.ln1       = nn.LayerNorm(config.d_model)
        self.retention = MultiScaleRetention(config)
        self.ln2       = nn.LayerNorm(config.d_model)
        self.ffn       = FeedForward(config)

    def forward(self, x, recurrent_states=None, position=0):
        # Retention sub-layer
        ret_out = self.retention(self.ln1(x), recurrent_states, position)
        if isinstance(ret_out, tuple):
            ret_out, new_states = ret_out
        else:
            new_states = None
        x = x + ret_out

        # FFN sub-layer
        x = x + self.ffn(self.ln2(x))

        if new_states is not None:
            return x, new_states
        return x


print("FeedForward + RetNetBlock defined.")

In [ ]:
# ────────────────────────────────────────────────────────────
#  Full RetNet Language Model — 124M parameters
# ────────────────────────────────────────────────────────────

class RetNetLM(nn.Module):
    """
    Training:  model(input_ids)      -> logits (B, T, vocab_size)
    Inference: model.generate(...)   -> token ids, using recurrent mode
    """

    def __init__(self, config: Optional[RetNetConfig] = None):
        super().__init__()
        if config is None:
            config = RetNetConfig()
        self.config = config

        self.embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.blocks    = nn.ModuleList([RetNetBlock(config) for _ in range(config.n_layers)])
        self.final_ln  = nn.LayerNorm(config.d_model)

        # LM head — weight-tied to embedding (saves ~39M params)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.02)
        # Scale residual projections by 1/sqrt(2 * n_layers) for stability
        scale = (2 * self.config.n_layers) ** -0.5
        for block in self.blocks:
            nn.init.normal_(block.retention.out_proj.weight, std=scale * 0.02)
            nn.init.normal_(block.ffn.fc2.weight,            std=scale * 0.02)

    def num_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    # ── Parallel forward (training) ──────────────────────────

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """input_ids: (B, T) -> logits (B, T, vocab_size)"""
        x = self.embedding(input_ids)   # (B, T, d_model)
        for block in self.blocks:
            x = block(x)                # parallel mode: returns tensor
        return self.lm_head(self.final_ln(x))

    # ── Recurrent single-step (inference) ────────────────────

    def forward_recurrent_step(
        self,
        token_id: torch.Tensor,
        all_states: Optional[List[List[torch.Tensor]]],
        position: int,
    ) -> Tuple[torch.Tensor, List[List[torch.Tensor]]]:
        """
        token_id:   (B,)
        all_states: list[n_layers] of list[n_heads] of (B, d_qk, d_v)
        Returns:    logits (B, vocab_size), new_all_states
        """
        x = self.embedding(token_id.unsqueeze(1))   # (B, 1, d_model)
        new_all_states = []
        for i, block in enumerate(self.blocks):
            states_i = all_states[i] if all_states is not None else None
            x, new_states_i = block(x, recurrent_states=states_i, position=position)
            new_all_states.append(new_states_i)
        logits = self.lm_head(self.final_ln(x.squeeze(1)))   # (B, vocab_size)
        return logits, new_all_states

    # ── Autoregressive generation ─────────────────────────────

    @torch.no_grad()
    def generate(
        self,
        prompt_ids: torch.Tensor,
        max_new_tokens: int = GEN_MAX_NEW_TOKENS,
        temperature:    float = GEN_TEMPERATURE,
        top_p:          float = GEN_TOP_P,
        repetition_penalty: float = GEN_REP_PENALTY,
        ngram_block:    int = GEN_NGRAM_BLOCK,
    ) -> torch.Tensor:
        """Autoregressive generation using recurrent mode (O(1) memory per step)."""
        self.eval()
        B, T_prompt = prompt_ids.shape
        device = prompt_ids.device

        # Prefill: process prompt token-by-token to build recurrent state
        all_states = None
        generated  = prompt_ids.clone()
        for t in range(T_prompt):
            _, all_states = self.forward_recurrent_step(generated[:, t], all_states, t)

        # Generate token by token
        for step in range(max_new_tokens):
            logits, all_states = self.forward_recurrent_step(
                generated[:, -1], all_states, T_prompt + step
            )   # logits: (B, vocab)

            # Repetition penalty
            if repetition_penalty != 1.0:
                for b in range(B):
                    for tid in set(generated[b].tolist()):
                        if logits[b, tid] > 0: logits[b, tid] /= repetition_penalty
                        else:                  logits[b, tid] *= repetition_penalty

            # N-gram blocking
            if ngram_block > 0 and generated.shape[1] >= ngram_block:
                for b in range(B):
                    last_ng = tuple(generated[b, -(ngram_block-1):].tolist())
                    hist    = generated[b].tolist()
                    blocked = {hist[i + ngram_block - 1]
                               for i in range(len(hist) - ngram_block + 1)
                               if tuple(hist[i:i+ngram_block-1]) == last_ng}
                    for tid in blocked: logits[b, tid] = float('-inf')

            # Temperature + top-p nucleus sampling
            logits = logits / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum = sorted_probs.cumsum(dim=-1)
            sorted_probs[cum - sorted_probs > top_p] = 0.0
            sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True).clamp(min=1e-10)
            next_sorted = torch.multinomial(sorted_probs, 1).squeeze(1)
            next_token  = sorted_idx.gather(1, next_sorted.unsqueeze(1)).squeeze(1)
            generated   = torch.cat([generated, next_token.unsqueeze(1)], dim=1)

        return generated


print("RetNetLM defined.")

In [ ]:
# ── Instantiate model and count parameters ──────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = RetNetConfig()
model  = RetNetLM(config).to(device)

n_params = model.num_parameters()
print(f"Parameters     : {n_params:,}  (~{n_params/1e6:.1f}M)")

# Quick architecture verification
model.eval()
with torch.no_grad():
    # Parallel (training) mode
    dummy_ids = torch.randint(0, config.vocab_size, (2, 64), device=device)
    logits = model(dummy_ids)
    print(f"Parallel mode  : {tuple(dummy_ids.shape)} -> {tuple(logits.shape)}")

    # Recurrent (inference) mode — single step
    tok    = torch.randint(0, config.vocab_size, (2,), device=device)
    lg, _  = model.forward_recurrent_step(tok, None, position=0)
    print(f"Recurrent mode : (B=2,) -> {tuple(lg.shape)}")

model.train()
print("Architecture OK.")

---
## 4. Dataset — WikiText-103

- **Download**: ~500MB via HuggingFace `datasets` (cached after first run)
- **Tokenizer**: GPT-2 tokenizer → vocab size 50,257
- **Chunking**: Flat token stream split into 512-token windows (no padding)
- **Result**: ~228K training chunks, ~478 val chunks, ~548 test chunks

In [ ]:
import os
from torch.utils.data import Dataset, DataLoader

CHUNK_SIZE = MAX_SEQ_LEN   # 512 tokens per sequence
DATA_CACHE = Path("./data_cache")
DATA_CACHE.mkdir(exist_ok=True)


class ChunkedTokenDataset(Dataset):
    """
    Flat list of token IDs -> (input, target) pairs of length CHUNK_SIZE.
    input[i]  = tokens[i*C : i*C + C]
    target[i] = tokens[i*C+1 : i*C + C + 1]   (next-token prediction)
    """
    def __init__(self, token_ids: torch.Tensor, chunk_size: int = CHUNK_SIZE):
        self.tokens     = token_ids
        self.chunk_size = chunk_size
        self.n_chunks   = (len(token_ids) - 1) // chunk_size

    def __len__(self):  return self.n_chunks

    def __getitem__(self, idx):
        s = idx * self.chunk_size
        return self.tokens[s:s+self.chunk_size], self.tokens[s+1:s+self.chunk_size+1]


def get_tokenizer():
    from transformers import GPT2TokenizerFast
    tok = GPT2TokenizerFast.from_pretrained("gpt2")
    tok.pad_token = tok.eos_token
    return tok


def build_datasets(verbose=True) -> dict:
    """
    Download, tokenize, and cache WikiText-103.
    Returns {'train': Dataset, 'val': Dataset, 'test': Dataset, 'stats': dict}
    """
    cache_file = DATA_CACHE / "wikitext103_gpt2.pt"

    if cache_file.exists():
        if verbose: print(f"Loading cached tokens from {cache_file}")
        data = torch.load(cache_file)
    else:
        if verbose: print("Downloading WikiText-103 (~500MB, first time only)...")
        from datasets import load_dataset
        raw = load_dataset("wikitext", "wikitext-103-raw-v1")
        tokenizer = get_tokenizer()
        data = {}

        for split_name, hf_split in [("train", "train"), ("val", "validation"), ("test", "test")]:
            if verbose: print(f"  Tokenizing {split_name}...")
            texts   = raw[hf_split]["text"]
            all_ids = []
            eos_id  = tokenizer.eos_token_id
            for text in texts:
                text = text.strip()
                if text:
                    all_ids.extend(tokenizer.encode(text, add_special_tokens=False))
                    all_ids.append(eos_id)
            data[split_name] = torch.tensor(all_ids, dtype=torch.long)
            if verbose: print(f"  {split_name}: {len(data[split_name]):,} tokens")

        torch.save(data, cache_file)
        if verbose: print(f"Saved cache to {cache_file}")

    train_ds = ChunkedTokenDataset(data["train"])
    val_ds   = ChunkedTokenDataset(data["val"])
    test_ds  = ChunkedTokenDataset(data["test"])

    stats = {
        "train_tokens": len(data["train"]), "val_tokens": len(data["val"]),
        "test_tokens":  len(data["test"]),  "train_chunks": len(train_ds),
        "val_chunks":   len(val_ds),        "test_chunks":  len(test_ds),
    }

    if verbose:
        print(f"Train: {len(train_ds):,} chunks | Val: {len(val_ds):,} | Test: {len(test_ds):,}")

    return {"train": train_ds, "val": val_ds, "test": test_ds, "stats": stats}


def get_dataloaders(batch_size: int = PHYS_BATCH) -> dict:
    datasets = build_datasets()
    kw = dict(pin_memory=True, num_workers=0, drop_last=True)
    return {
        "train": DataLoader(datasets["train"], batch_size=batch_size, shuffle=True,  **kw),
        "val":   DataLoader(datasets["val"],   batch_size=batch_size, shuffle=False, **kw),
        "test":  DataLoader(datasets["test"],  batch_size=batch_size, shuffle=False, **kw),
        "stats": datasets["stats"],
    }


print("Dataset classes defined. Run the next cell to load data.")

In [ ]:
# Load WikiText-103  (first run: ~5 min download + tokenize; subsequent runs: instant)
loaders = get_dataloaders(batch_size=PHYS_BATCH)

stats = loaders["stats"]
print("\n=" * 40)
print("  Dataset Stats")
print("=" * 40)
print(f"  Train : {stats['train_tokens']:>12,} tokens | {stats['train_chunks']:>8,} chunks")
print(f"  Val   : {stats['val_tokens']:>12,} tokens | {stats['val_chunks']:>8,} chunks")
print(f"  Test  : {stats['test_tokens']:>12,} tokens | {stats['test_chunks']:>8,} chunks")
print(f"  Chunk size : {CHUNK_SIZE} tokens  |  Vocab: {VOCAB_SIZE:,}")

# Peek at a batch
x, y = next(iter(loaders["train"]))
print(f"\nBatch shape  : x={tuple(x.shape)}, y={tuple(y.shape)}")
print(f"x[0,:6]      : {x[0,:6].tolist()}")
print(f"y[0,:6]      : {y[0,:6].tolist()}  <- shifted by 1")

---
## 5. Training Setup

### LR Schedule: Warmup → Cosine Decay
```
Steps 0–1000     : Linear warmup  0 → 3e-4
Steps 1000–50000 : Cosine decay   3e-4 → 3e-5
```

### Optimizer: AdamW
- β1=0.9, β2=0.98 (standard for transformers)
- weight_decay=0.05 on weight matrices only (not biases, LayerNorm, embeddings)

### FP16 Mixed Precision
- `torch.cuda.amp.autocast` for forward passes
- `GradScaler` to prevent gradient underflow

### Gradient Accumulation
- Physical batch=8 → accumulate 4 steps → effective batch=32
- Needed to fit in 8GB VRAM while maintaining stable training dynamics

In [ ]:
import time, math, csv, json
from datetime import datetime, timedelta
from pathlib import Path
from torch.cuda.amp import GradScaler, autocast

CKPT_DIR = Path(CHECKPOINT_DIR)
CKPT_DIR.mkdir(exist_ok=True)


# ── Learning Rate Schedule ───────────────────────────────────

def get_lr(step: int) -> float:
    """
    Linear warmup for WARMUP_STEPS, then cosine decay to MAX_LR * MIN_LR_RATIO.
    """
    if step < WARMUP_STEPS:
        return MAX_LR * step / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / (TOTAL_STEPS - WARMUP_STEPS)
    progress = min(progress, 1.0)
    cosine_val = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MAX_LR * MIN_LR_RATIO + (MAX_LR - MAX_LR * MIN_LR_RATIO) * cosine_val


# ── Evaluation ───────────────────────────────────────────────

@torch.no_grad()
def evaluate(model, loader, max_batches=None) -> float:
    """Compute perplexity on a dataloader. Returns PPL."""
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for i, (x, y) in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        with autocast():
            logits = model(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1), reduction="sum")
        total_loss   += loss.item()
        total_tokens += y.numel()
    avg_loss = total_loss / total_tokens
    model.train()
    return math.exp(min(avg_loss, 20.0))  # clamp to avoid overflow


# ── Optimizer ────────────────────────────────────────────────

# Split params: apply weight decay only to weight matrices
decay_params, no_decay_params = [], []
for name, p in model.named_parameters():
    if not p.requires_grad: continue
    if any(k in name for k in ["bias", "ln", "norm", "embedding"]):
        no_decay_params.append(p)
    else:
        decay_params.append(p)

optimizer = torch.optim.AdamW(
    [{"params": decay_params,    "weight_decay": WEIGHT_DECAY},
     {"params": no_decay_params, "weight_decay": 0.0}],
    lr=MAX_LR, betas=(ADAM_BETA1, ADAM_BETA2), eps=ADAM_EPS
)
scaler = GradScaler()

print(f"Optimizer     : AdamW | {sum(p.numel() for p in decay_params):,} params with wd")
print(f"               {sum(p.numel() for p in no_decay_params):,} params without wd")


# ── Resume from checkpoint if exists ─────────────────────────

start_step   = 0
best_val_ppl = float("inf")
ckpt_latest  = CKPT_DIR / "latest.pt"

if ckpt_latest.exists():
    print(f"\nResuming from {ckpt_latest}...")
    ckpt = torch.load(ckpt_latest, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    start_step   = ckpt["step"]
    best_val_ppl = ckpt.get("best_val_ppl", float("inf"))
    print(f"Resumed from step {start_step:,} | Best val PPL so far: {best_val_ppl:.2f}")
else:
    print("\nNo checkpoint found — starting fresh.")

---
## 6. Training Loop

This cell runs the full 50,000-step training.  
**Expected time**: ~24 hours on RTX 2070 (~0.5 steps/sec)  
**On Colab T4**: ~12-16 hours

Metrics printed every 50 steps. Validation every 2,000 steps.  
Checkpoints saved every 2,000 steps to `./checkpoints/`.  
The cell is safe to interrupt and re-run — it resumes automatically.

In [ ]:
# ── CSV Logger ───────────────────────────────────────────────

log_file   = open(CKPT_DIR / "training_log.csv", "w", newline="", encoding="utf-8")
csv_writer = csv.writer(log_file)
csv_writer.writerow(["step", "train_loss", "train_ppl", "val_ppl", "lr", "gpu_gb", "elapsed_s"])

def log_csv(step, train_loss, train_ppl, val_ppl, lr, elapsed):
    gpu_gb = torch.cuda.memory_allocated(0) / 1024**3 if device.type == "cuda" else 0.0
    csv_writer.writerow([step, f"{train_loss:.4f}", f"{train_ppl:.2f}",
                         f"{val_ppl:.2f}" if val_ppl else "",
                         f"{lr:.6f}", f"{gpu_gb:.2f}", f"{elapsed:.0f}"])
    log_file.flush()


# ── Training State JSON (for external monitoring) ────────────

val_history = []

def save_state(step, loss, ppl, lr, eta_str, phase):
    gpu_gb = torch.cuda.memory_allocated(0) / 1024**3 if device.type == "cuda" else 0.0
    d = {
        "step": step, "total_steps": TOTAL_STEPS,
        "loss": round(loss, 4) if math.isfinite(loss) else None,
        "ppl":  round(ppl,  2) if math.isfinite(ppl)  else None,
        "best_val_ppl": round(best_val_ppl, 2) if math.isfinite(best_val_ppl) else None,
        "lr": f"{lr:.2e}", "gpu_gb": round(gpu_gb, 2),
        "eta": eta_str, "phase": phase,
        "val_history": val_history,
        "updated_at": datetime.now().isoformat(),
    }
    with open(CKPT_DIR / "training_state.json", "w") as f:
        json.dump(d, f, indent=2)


# ── Main Training Loop ───────────────────────────────────────

model.train()
train_iter   = iter(loaders["train"])
global_step  = start_step
micro_count  = 0
running_loss = 0.0
step_times   = []
t0           = time.time()
t_step       = time.time()

optimizer.zero_grad()

print(f"Training starts at step {global_step:,} / {TOTAL_STEPS:,}")
print(f"Physical batch={PHYS_BATCH}, accum={ACCUM_STEPS}, effective batch={EFFECTIVE_BATCH}")
print("-" * 70)
print(f"{'Step':>8}  {'Loss':>8}  {'PPL':>8}  {'Val PPL':>9}  {'LR':>10}  {'GPU GB':>7}  ETA")
print("-" * 70)

try:
    while global_step < TOTAL_STEPS:
        # Fetch next batch (restart iterator when exhausted)
        try:
            x_b, y_b = next(train_iter)
        except StopIteration:
            train_iter = iter(loaders["train"])
            x_b, y_b = next(train_iter)

        x_b = x_b.to(device, non_blocking=True)
        y_b = y_b.to(device, non_blocking=True)

        # Forward pass in FP16
        with autocast():
            logits = model(x_b)   # (B, T, vocab)
            loss   = F.cross_entropy(logits.view(-1, logits.size(-1)), y_b.view(-1))
            loss_s = loss / ACCUM_STEPS

        scaler.scale(loss_s).backward()
        running_loss += loss.item()
        micro_count  += 1

        # ── Gradient update every ACCUM_STEPS micro-batches ──
        if micro_count % ACCUM_STEPS == 0:

            # Unscale + clip
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

            # Update LR
            lr = get_lr(global_step)
            for pg in optimizer.param_groups: pg["lr"] = lr

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            global_step += 1

            # Timing
            now = time.time()
            step_times.append(now - t_step)
            if len(step_times) > 100: step_times.pop(0)
            t_step = now

            avg_step_t = sum(step_times) / len(step_times) if step_times else 1.0
            eta_str    = str(timedelta(seconds=int(avg_step_t * (TOTAL_STEPS - global_step))))

            avg_loss     = running_loss / ACCUM_STEPS
            running_loss = 0.0
            ppl          = math.exp(min(avg_loss, 20.0))
            phase        = "Warmup" if global_step < WARMUP_STEPS else "Cosine Decay"

            # ── Print + CSV every LOG_EVERY steps ──
            if global_step % LOG_EVERY == 0:
                gpu_gb  = torch.cuda.memory_allocated(0)/1024**3 if device.type=="cuda" else 0
                elapsed = now - t0
                log_csv(global_step, avg_loss, ppl, None, lr, elapsed)
                speed   = 1.0 / avg_step_t
                print(f"{global_step:>8,}  {avg_loss:>8.4f}  {ppl:>8.2f}  {'---':>9}  {lr:>10.2e}  {gpu_gb:>7.2f}  {eta_str}")

            # ── Save JSON state every 100 steps ──
            if global_step % 100 == 0:
                save_state(global_step, avg_loss, ppl, lr, eta_str, phase)

            # ── Validation every EVAL_EVERY steps ──
            if global_step % EVAL_EVERY == 0 or global_step == TOTAL_STEPS:
                torch.cuda.empty_cache()   # free fragmented VRAM
                val_ppl = evaluate(model, loaders["val"])
                is_best = val_ppl < best_val_ppl
                if is_best: best_val_ppl = val_ppl
                val_history.append((global_step, round(val_ppl, 2)))

                elapsed = time.time() - t0
                log_csv(global_step, avg_loss, ppl, val_ppl, lr, elapsed)
                save_state(global_step, avg_loss, ppl, lr, eta_str, phase)

                marker = "  <-- NEW BEST" if is_best else ""
                print(f"{'':->8}  {'':->8}  {'':->8}  {val_ppl:>9.2f}  {'[VAL]':>10}  {'':>7}  {marker}")

                # Save best model
                if is_best:
                    torch.save({
                        "step": global_step, "model": model.state_dict(),
                        "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                        "val_ppl": val_ppl, "best_val_ppl": best_val_ppl,
                        "config": config.__dict__,
                    }, CKPT_DIR / "best_model.pt")

                model.train()

            # ── Periodic checkpoint every SAVE_EVERY steps ──
            if global_step % SAVE_EVERY == 0:
                torch.save({
                    "step": global_step, "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                    "val_ppl": best_val_ppl, "best_val_ppl": best_val_ppl,
                    "config": config.__dict__,
                }, CKPT_DIR / "latest.pt")
                print(f"  [Checkpoint saved at step {global_step:,}]")

except KeyboardInterrupt:
    print("\nTraining interrupted. Saving checkpoint...")
    torch.save({
        "step": global_step, "model": model.state_dict(),
        "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
        "val_ppl": best_val_ppl, "best_val_ppl": best_val_ppl,
        "config": config.__dict__,
    }, CKPT_DIR / "latest.pt")
    print(f"  Saved at step {global_step:,} to {CKPT_DIR / 'latest.pt'}")

log_file.close()
print("\nTraining loop exited.")

---
## 7. Final Evaluation

Evaluate the **best saved checkpoint** on the held-out WikiText-103 test set.  
Target perplexity: **~15.2** (paper result).

In [ ]:
# Load best checkpoint and evaluate on test set

best_ckpt = CKPT_DIR / "best_model.pt"

if best_ckpt.exists():
    print(f"Loading best checkpoint: {best_ckpt}")
    ckpt = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(ckpt["model"])
    val_ppl_at_save = ckpt.get("val_ppl", "?")
    saved_at_step   = ckpt.get("step", "?")
    print(f"  Saved at step {saved_at_step} | Val PPL at save time: {val_ppl_at_save}")
else:
    print("No best_model.pt found — evaluating current model state.")

print("\nRunning full test set evaluation...")
test_ppl = evaluate(model, loaders["test"])

print()
print("=" * 50)
print("  FINAL EVALUATION — WikiText-103 Test Set")
print("=" * 50)
print(f"  Test Perplexity : {test_ppl:.2f}")
print(f"  Paper target    : ~15.2")
if   test_ppl <= 20.0: print("  Status: EXCELLENT — within target range!")
elif test_ppl <= 30.0: print("  Status: GOOD — close to target.")
else:                  print("  Status: May benefit from more training steps.")
print("=" * 50)

# Save final model with test PPL recorded
torch.save({
    "step": global_step, "model": model.state_dict(),
    "config": config.__dict__,
    "val_ppl": best_val_ppl, "test_ppl": test_ppl,
}, CKPT_DIR / "final_model.pt")
print(f"\nFinal model saved to {CKPT_DIR / 'final_model.pt'}")

---
## 8. Text Generation

Uses the **recurrent mode** — O(1) memory per token step.  
The model processes the prompt token-by-token (prefill), then generates autoregressively.

> **Perplexity-quality mismatch**: This model achieves low perplexity on next-token prediction but has not been fine-tuned with RLHF/instruction tuning. Expect grammatically plausible but potentially semantically drifting text.

In [ ]:
# Generate text from a prompt

tokenizer = get_tokenizer()

def generate_text(
    prompt: str,
    max_new_tokens: int = GEN_MAX_NEW_TOKENS,
    temperature:    float = GEN_TEMPERATURE,
    top_p:          float = GEN_TOP_P,
    rep_penalty:    float = GEN_REP_PENALTY,
    ngram_block:    int   = GEN_NGRAM_BLOCK,
) -> str:
    prompt_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    output_ids = model.generate(
        prompt_ids, max_new_tokens=max_new_tokens,
        temperature=temperature, top_p=top_p,
        repetition_penalty=rep_penalty, ngram_block=ngram_block,
    )
    # Decode only the newly generated tokens
    generated_ids = output_ids[0, len(prompt_ids[0]):].tolist()
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


# ── Single prompt ────────────────────────────────────────────
prompt = "The history of artificial intelligence"
print(f"[Prompt]    {prompt}")
print(f"[Mode]      Recurrent (O(1) memory per step)")
print(f"[Params]    temp={GEN_TEMPERATURE}, top_p={GEN_TOP_P}, rep_pen={GEN_REP_PENALTY}")
print()

text = generate_text(prompt)
print(f"[Generated] {text}")

In [ ]:
# Temperature sweep — shows the effect of temperature on output diversity

prompt = "In the beginning, scientists discovered"
print(f"Prompt: '{prompt}'\n")
print("=" * 60)

for temp in [0.5, 0.8, 1.0, 1.2]:
    text = generate_text(prompt, max_new_tokens=80, temperature=temp)
    print(f"\n[temp={temp}]")
    print(text[:250])
    print("-" * 60)

In [ ]:
# ── Summary ──────────────────────────────────────────────────

print("=" * 60)
print("  TRAINING COMPLETE — SUMMARY")
print("=" * 60)
print(f"  Model          : RetNet 124M (d={D_MODEL}, L={N_LAYERS}, H={N_HEADS})")
print(f"  Parameters     : {model.num_parameters():,}")
print(f"  Total steps    : {TOTAL_STEPS:,}")
print(f"  Best Val PPL   : {best_val_ppl:.2f}")
try:
    print(f"  Test PPL       : {test_ppl:.2f}")
except NameError:
    print("  Test PPL       : (run evaluation cell)")
print(f"  Checkpoints    : {CKPT_DIR}")
print(f"    - best_model.pt   <- best validation checkpoint")
print(f"    - final_model.pt  <- end-of-training checkpoint")
print(f"    - latest.pt       <- most recent periodic checkpoint")
print(f"    - training_log.csv")
print("=" * 60)

---
## Part 2 — Supervised Fine-Tuning (SFT) on Stanford Alpaca

Fine-tunes the pretrained 124M RetNet on **52K instruction-response pairs** from Stanford Alpaca so the model learns to follow the Alpaca prompt format.

Key design choices:
- **Instruction masking** — loss is computed only on Response tokens, not the instruction
- **Small LR** (5e-5 vs 3e-4 during pretraining) to preserve pretrained weights
- **800 steps** (~0.26 epochs) to avoid memorisation
- `generate_parallel` mode at inference — uses the same full-forward computation as training

In [ ]:
# ── SFT Hyperparameters ───────────────────────────────────────────────────────
import math, csv, json as _json

SFT_TOTAL_STEPS  = 800        # ~0.26 epochs — avoids memorisation
SFT_WARMUP_STEPS = 80         # 10% warmup
SFT_PHYS_BATCH   = 4          # physical batch size
SFT_ACCUM_STEPS  = 4          # gradient accumulation → effective batch = 16
SFT_MAX_LR       = 5e-5       # 6× smaller than pretraining LR
SFT_GRAD_CLIP    = 1.0
SFT_EVAL_EVERY   = 100
SFT_MAX_SEQ_LEN  = 512

SFT_CKPT_DIR = Path("checkpoints_sft")
SFT_CKPT_DIR.mkdir(exist_ok=True)

ALPACA_WITH_INPUT = """\
### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

ALPACA_NO_INPUT = """\
### Instruction:
{instruction}

### Response:
{output}"""

RESPONSE_SPLIT = "### Response:\n"

print(f"SFT config: {SFT_TOTAL_STEPS} steps | LR={SFT_MAX_LR} | batch={SFT_PHYS_BATCH}x{SFT_ACCUM_STEPS}={SFT_PHYS_BATCH*SFT_ACCUM_STEPS} effective")

In [ ]:
# ── AlpacaDataset — tokenise with instruction masking ────────────────────────

class AlpacaDataset(Dataset):
    """
    Tokenises Alpaca samples. Labels are -100 for instruction tokens so
    cross-entropy loss is only computed on the Response portion.
    """
    def __init__(self, samples, tokenizer, max_len=512):
        self.items = []
        eos      = tokenizer.eos_token_id
        resp_ids = tokenizer.encode(RESPONSE_SPLIT, add_special_tokens=False)

        for s in samples:
            if s.get("input", "").strip():
                text = ALPACA_WITH_INPUT.format(**s)
            else:
                text = ALPACA_NO_INPUT.format(instruction=s["instruction"], output=s["output"])

            full_ids = tokenizer.encode(text, add_special_tokens=False) + [eos]
            if len(full_ids) > max_len:
                full_ids = full_ids[:max_len]

            # Find start of response text
            resp_start = next(
                (i + len(resp_ids) for i in range(len(full_ids) - len(resp_ids) + 1)
                 if full_ids[i:i+len(resp_ids)] == resp_ids), 0
            )

            input_ids = full_ids[:-1]
            labels    = [-100] * len(input_ids)
            for i in range(resp_start, len(input_ids)):
                labels[i] = input_ids[i]

            pad = (max_len - 1) - len(input_ids)
            input_ids += [eos] * pad
            labels    += [-100] * pad

            self.items.append((
                torch.tensor(input_ids, dtype=torch.long),
                torch.tensor(labels,    dtype=torch.long),
            ))

    def __len__(self):  return len(self.items)
    def __getitem__(self, i): return self.items[i]


# ── Load Stanford Alpaca ──────────────────────────────────────────────────────

import json as _json
from pathlib import Path

alpaca_cache = Path("data_cache/alpaca.json")
alpaca_cache.parent.mkdir(exist_ok=True)

if alpaca_cache.exists():
    print("Loading cached Alpaca dataset...")
    alpaca_data = _json.loads(alpaca_cache.read_text())
else:
    print("Downloading Stanford Alpaca (52K samples)...")
    from datasets import load_dataset
    ds = load_dataset("tatsu-lab/alpaca", split="train")
    alpaca_data = [{"instruction": r["instruction"], "input": r["input"], "output": r["output"]} for r in ds]
    alpaca_cache.write_text(_json.dumps(alpaca_data))

alpaca_data = [s for s in alpaca_data if s["output"].strip()]
split_idx   = int(len(alpaca_data) * 0.95)
train_data  = alpaca_data[:split_idx]
val_data    = alpaca_data[split_idx:]

print(f"Tokenising... ({len(train_data):,} train / {len(val_data):,} val)")
sft_train_ds = AlpacaDataset(train_data, tokenizer, SFT_MAX_SEQ_LEN)
sft_val_ds   = AlpacaDataset(val_data,   tokenizer, SFT_MAX_SEQ_LEN)

sft_train_loader = DataLoader(sft_train_ds, batch_size=SFT_PHYS_BATCH, shuffle=True,  pin_memory=True, num_workers=0, drop_last=True)
sft_val_loader   = DataLoader(sft_val_ds,   batch_size=SFT_PHYS_BATCH, shuffle=False, pin_memory=True, num_workers=0)

print(f"Done — {len(sft_train_ds):,} train samples | {len(sft_val_ds):,} val samples")

In [ ]:
# ── Load pretrained base model for SFT ───────────────────────────────────────

sft_ckpt_path = Path("checkpoints/best_model.pt")
assert sft_ckpt_path.exists(), "Run pretraining cells first to generate checkpoints/best_model.pt"

sft_ckpt   = torch.load(sft_ckpt_path, map_location=DEVICE)
sft_config = RetNetConfig()
if "config" in sft_ckpt:
    for k, v in sft_ckpt["config"].items():
        if hasattr(sft_config, k): setattr(sft_config, k, v)

sft_model = RetNetLM(sft_config).to(DEVICE)
sft_model.load_state_dict(sft_ckpt["model"])
print(f"Loaded base model: {sft_model.num_parameters():,} params | pretrain val PPL={sft_ckpt.get('val_ppl','?')}")

# ── Optimizer ─────────────────────────────────────────────────────────────────

decay_p, no_decay_p = [], []
for name, p in sft_model.named_parameters():
    if not p.requires_grad: continue
    if any(k in name for k in ["bias", "ln", "norm", "embedding"]):
        no_decay_p.append(p)
    else:
        decay_p.append(p)

sft_optimizer = torch.optim.AdamW(
    [{"params": decay_p,    "weight_decay": 0.01},
     {"params": no_decay_p, "weight_decay": 0.0}],
    lr=SFT_MAX_LR, betas=(0.9, 0.98), eps=1e-8,
)
sft_scaler = torch.cuda.amp.GradScaler()

def sft_lr(step):
    if step < SFT_WARMUP_STEPS:
        return SFT_MAX_LR * step / max(SFT_WARMUP_STEPS, 1)
    progress = (step - SFT_WARMUP_STEPS) / max(SFT_TOTAL_STEPS - SFT_WARMUP_STEPS, 1)
    cosine   = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
    return SFT_MAX_LR * 0.1 + (SFT_MAX_LR - SFT_MAX_LR * 0.1) * cosine

print("Optimizer ready.")

In [ ]:
# ── SFT Training Loop ─────────────────────────────────────────────────────────

@torch.no_grad()
def sft_evaluate(model, loader, device, max_batches=None):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for i, (x, y) in enumerate(loader):
        if max_batches and i >= max_batches: break
        x, y = x.to(device), y.to(device)
        with torch.cuda.amp.autocast():
            logits = model(x)
        loss  = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1),
                                ignore_index=-100, reduction="sum")
        n_tok = (y != -100).sum().item()
        if n_tok > 0:
            total_loss   += loss.item()
            total_tokens += n_tok
    model.train()
    return math.exp(min(total_loss / max(total_tokens, 1), 20.0))


sft_model.train()
sft_iter      = iter(sft_train_loader)
sft_step      = 0
sft_micro     = 0
sft_run_loss  = 0.0
sft_best_ppl  = float("inf")
sft_val_hist  = []
sft_log_csv   = open(SFT_CKPT_DIR / "finetune_log.csv", "w", newline="")
csv_w         = csv.writer(sft_log_csv)
csv_w.writerow(["step", "train_loss", "val_ppl"])

sft_optimizer.zero_grad()
print(f"Starting SFT — {SFT_TOTAL_STEPS} steps\n")

while sft_step < SFT_TOTAL_STEPS:
    try:
        xb, yb = next(sft_iter)
    except StopIteration:
        sft_iter = iter(sft_train_loader)
        xb, yb  = next(sft_iter)

    xb, yb = xb.to(DEVICE), yb.to(DEVICE)

    with torch.cuda.amp.autocast():
        logits = sft_model(xb)
        loss   = F.cross_entropy(logits.view(-1, logits.size(-1)), yb.view(-1), ignore_index=-100)
        loss_s = loss / SFT_ACCUM_STEPS

    sft_scaler.scale(loss_s).backward()
    sft_run_loss += loss.item()
    sft_micro    += 1

    if sft_micro % SFT_ACCUM_STEPS == 0:
        sft_scaler.unscale_(sft_optimizer)
        torch.nn.utils.clip_grad_norm_(sft_model.parameters(), SFT_GRAD_CLIP)

        lr = sft_lr(sft_step)
        for pg in sft_optimizer.param_groups: pg["lr"] = lr

        sft_scaler.step(sft_optimizer)
        sft_scaler.update()
        sft_optimizer.zero_grad()
        sft_step += 1

        avg_loss  = sft_run_loss / SFT_ACCUM_STEPS
        sft_run_loss = 0.0
        ppl       = math.exp(min(avg_loss, 20.0))

        if sft_step % 50 == 0:
            print(f"  step {sft_step:>4}/{SFT_TOTAL_STEPS} | loss={avg_loss:.3f} | ppl={ppl:.2f} | lr={lr:.1e}")

        if sft_step % SFT_EVAL_EVERY == 0 or sft_step == SFT_TOTAL_STEPS:
            val_ppl  = sft_evaluate(sft_model, sft_val_loader, DEVICE)
            is_best  = val_ppl < sft_best_ppl
            if is_best: sft_best_ppl = val_ppl
            sft_val_hist.append((sft_step, round(val_ppl, 2)))
            csv_w.writerow([sft_step, f"{avg_loss:.4f}", f"{val_ppl:.2f}"])
            sft_log_csv.flush()

            marker = " ← best" if is_best else ""
            print(f"  [val] step {sft_step} | val_ppl={val_ppl:.2f}{marker}")

            if is_best:
                torch.save({"step": sft_step, "model": sft_model.state_dict(),
                            "val_ppl": val_ppl, "config": sft_config.__dict__},
                           SFT_CKPT_DIR / "best_sft.pt")

sft_log_csv.close()
print(f"\nSFT complete! Best val PPL = {sft_best_ppl:.2f}")
print(f"Val history: {sft_val_hist}")

In [ ]:
# ── SFT Generation ────────────────────────────────────────────────────────────
# Uses generate_parallel — same full-forward computation as training,
# avoiding the GroupNorm statistics mismatch of recurrent mode.

sft_gen_ckpt = torch.load(SFT_CKPT_DIR / "best_sft.pt", map_location=DEVICE)
sft_gen_cfg  = RetNetConfig()
if "config" in sft_gen_ckpt:
    for k, v in sft_gen_ckpt["config"].items():
        if hasattr(sft_gen_cfg, k): setattr(sft_gen_cfg, k, v)

sft_gen_model = RetNetLM(sft_gen_cfg).to(DEVICE)
sft_gen_model.load_state_dict(sft_gen_ckpt["model"])
sft_gen_model.eval()
print(f"SFT model loaded — step={sft_gen_ckpt.get('step','?')} | val_ppl={sft_gen_ckpt.get('val_ppl','?'):.3f}\n")


@torch.no_grad()
def sft_generate(instruction, input_text="", max_new_tokens=200, temperature=0.7, top_p=0.9, rep_penalty=1.3):
    if input_text.strip():
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long).to(DEVICE)
    out = sft_gen_model.generate_parallel(
        ids, max_new_tokens=max_new_tokens, temperature=temperature,
        top_p=top_p, repetition_penalty=rep_penalty, ngram_block=3,
    )
    new_ids  = out[0, ids.shape[1]:].tolist()
    response = tokenizer.decode(new_ids, skip_special_tokens=True)
    for sep in ["\n\n", "###"]:
        if sep in response:
            response = response[:response.index(sep)]
    return response.strip()


# Run demo prompts
demos = [
    ("Explain what a neural network is in simple terms.", ""),
    ("Write a short poem about the ocean.", ""),
    ("Summarize the following text in one sentence.",
     "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France."),
    ("What are three tips for staying productive while working from home?", ""),
]

print("=" * 60)
for instr, inp in demos:
    resp = sft_generate(instr, inp)
    print(f"INSTRUCTION: {instr}")
    if inp: print(f"INPUT:       {inp}")
    print(f"RESPONSE:    {resp}")
    print("-" * 60)